In [1]:
import os, sys, re, json, glob, random, time, math, zipfile, shutil, gc
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
os.environ["TOKENIZERS_PARALLELISM"] = "false"

SMOKE = False      # FIRST run with True (~5 min, tiny subset). Then set False, Restart & Run All.
INPUT_ROOT = "/kaggle/input"
OUT = "/kaggle/working/" + ("smoke" if SMOKE else "run")
os.makedirs(OUT, exist_ok=True)

SEED = 42
CHUNK_LEN, STRIDE = 512, 384
MAX_CHUNKS = 3
EPOCHS = 1
TRAIN_CAP = None if SMOKE else 3000        # OOM -> 6 ; too slow -> 6
BATCH_DOCS, GRAD_ACCUM = 2, 2
HEAD_LR = 1e-3
LR = dict(lorma=5e-4, full=1e-4, bitfit=2e-3, kd=1e-4)
ID_KEY, PRED_KEY = "id", "pred"       # check spec Sec. 10

DEV = "cuda" if torch.cuda.is_available() else "cpu"
NUM_WORKERS = 2 if DEV == "cuda" else 0
def AC(): return torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(DEV == "cuda"))
def seed_all(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all()
print("device:", DEV, torch.cuda.get_device_name(0) if DEV == "cuda" else "", "| OUT:", OUT)

device: cuda Tesla T4 | OUT: /kaggle/working/run


In [2]:
from huggingface_hub import HfApi, hf_hub_download
from kaggle_secrets import UserSecretsClient
tok = UserSecretsClient().get_secret("HF_TOKEN")
print("len:", len(tok), "| starts with hf_:", tok.startswith("hf_"))
print("whoami:", HfApi(token=tok).whoami()["name"])
hf_hub_download("Exploration-Lab/CS-781-Capstone", "Phase_4_CJPE/Test/test.json",
                repo_type="dataset", local_dir="/kaggle/temp/hf_p4", token=tok)
print("download OK")

len: 37 | starts with hf_: True
whoami: S-240634


Phase_4_CJPE/Test/test.json: reconstructing file:   0%|          |  0.00B / 35.3MB            

Phase_4_CJPE/Test/test.json: downloading bytes:           |  0.00B            

download OK


In [3]:
from huggingface_hub import HfApi, hf_hub_download
HF_TOKEN = os.environ.get("HF_TOKEN")
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    print("no Kaggle secret HF_TOKEN:", type(e).__name__)

REPO, RAW = "Exploration-Lab/CS-781-Capstone", "/kaggle/temp/hf_p4"
os.makedirs(RAW, exist_ok=True)
files = HfApi(token=HF_TOKEN).list_repo_files(REPO, repo_type="dataset")
p4 = [f for f in files if any(k in f.lower() for k in ("phase_4", "phase4", "phase 4", "cjpe"))
      and f.lower().endswith((".json", ".jsonl", ".csv", ".tsv", ".parquet"))]
print("\n".join(p4)); assert p4, "No Phase 4 files found. Print `files` and send me the folder names."
local = {f: hf_hub_download(REPO, f, repo_type="dataset", local_dir=RAW, token=HF_TOKEN) for f in p4}

def pick_file(keys):
    c = [f for f in local if any(k in os.path.basename(f).lower() for k in keys)]
    return local[c[0]] if c else None
PATHS = dict(train=pick_file(("train",)), dev=pick_file(("dev", "val")), test=pick_file(("test",)))
print(PATHS)   # if wrong, overwrite PATHS by hand before Cell 3

Phase_4_CJPE/Test/test.json
Phase_4_CJPE/Train/dev.json
Phase_4_CJPE/Train/train.json


Phase_4_CJPE/Train/dev.json: reconstructing file:   0%|          |  0.00B / 45.0MB            

Phase_4_CJPE/Train/dev.json: downloading bytes:           |  0.00B            

Phase_4_CJPE/Train/train.json: reconstructing file:   0%|          |  0.00B /  737MB            

Phase_4_CJPE/Train/train.json: downloading bytes:           |  0.00B            

{'train': '/kaggle/temp/hf_p4/Phase_4_CJPE/Train/train.json', 'dev': '/kaggle/temp/hf_p4/Phase_4_CJPE/Train/dev.json', 'test': '/kaggle/temp/hf_p4/Phase_4_CJPE/Test/test.json'}


In [4]:
def read_any(p):
    pl = p.lower()
    if pl.endswith(".jsonl"):
        return [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]
    if pl.endswith(".csv"):     return pd.read_csv(p).to_dict("records")
    if pl.endswith(".tsv"):     return pd.read_csv(p, sep="\t").to_dict("records")
    if pl.endswith(".parquet"): return pd.read_parquet(p).to_dict("records")
    d = json.load(open(p, encoding="utf-8"))
    if isinstance(d, dict):
        lists = [v for v in d.values() if isinstance(v, list)]
        if len(d) <= 3 and lists: return max(lists, key=len)
        return [({**v, "_key": k} if isinstance(v, dict) else {"_key": k, "text": v}) for k, v in d.items()]
    return d

TEXT_KEYS = ("text", "document", "judgment", "case_text", "content", "full_text", "doc")
ID_KEYS   = ("id", "ID", "case_id", "doc_id", "_key", "index")
LAB_KEYS  = ("label", "labels", "decision", "outcome", "verdict", "y", "gold", "target")
def _pick(x, keys):
    for k in keys:
        if k in x and x[k] is not None and not (isinstance(x[k], float) and math.isnan(x[k])): return k
def norm(data):
    out = []
    for i, x in enumerate(data):
        tk = _pick(x, TEXT_KEYS) or max((k for k, v in x.items() if isinstance(v, (str, list))), key=lambda k: len(str(x[k])))
        ik, lk = _pick(x, ID_KEYS), _pick(x, LAB_KEYS)
        t = x[tk]; t = "\n".join(map(str, t)) if isinstance(t, list) else str(t)
        _id = x[ik] if ik else i
        if isinstance(_id, np.generic): _id = _id.item()
        out.append(dict(id=_id, text=t, label=x[lk] if lk else None))
    return out

train = norm(read_any(PATHS["train"]))
test  = norm(read_any(PATHS["test"]))
dev   = norm(read_any(PATHS["dev"])) if PATHS["dev"] else None
print("raw keys example:", list(read_any(PATHS["train"])[0].keys()))

labs = sorted({x["label"] for x in train if x["label"] is not None}, key=str)
assert labs, "No labels found in train. Check LAB_KEYS."
LABMAP = {0: 0, 1: 1, False: 0, True: 1} if set(labs) <= {0, 1, False, True} else {v: i for i, v in enumerate(labs)}
print("label map:", LABMAP); assert len(set(LABMAP.values())) == 2, "expected binary labels"
for d in (train, dev or []):
    for x in d: x["label"] = LABMAP[x["label"]]
if dev is None:
    rng = random.Random(SEED); rng.shuffle(train); k = len(train) // 10; dev, train = train[:k], train[k:]
if not SMOKE and TRAIN_CAP:
    rng = random.Random(SEED); rng.shuffle(train); train = train[:TRAIN_CAP]
if SMOKE:
    sub = lambda d, n: [x for c in (0, 1) for x in [y for y in d if y["label"] == c][:n]]
    train, dev, test = sub(train, 24), sub(dev, 12), test[:24]

print(len(train), len(dev), len(test), "| train labels", np.bincount([x["label"] for x in train]), "| dev labels", np.bincount([x["label"] for x in dev]))
w = [len(x["text"].split()) for x in train[:500]]; print("words median", np.median(w), "p90", np.percentile(w, 90))
print("example id:", train[0]["id"], "| test id:", test[0]["id"], "| test has labels:", test[0]["label"] is not None)

raw keys example: ['id', 'text', 'label', 'expert_1', 'expert_2', 'expert_3', 'expert_4', 'expert_5']
label map: {0: 0, 1: 1}
3000 1988 1517 | train labels [1750 1250] | dev labels [994 994]
words median 2147.0 p90 5884.6
example id: 1986_160 | test id: 1986_397 | test has labels: True


In [5]:
VOCAB_SIZE, PAD_ID, UNK_ID, BOS_ID, EOS_ID = 32000, 0, 1, 2, 3
DIM, N_LAYERS, N_HEADS = 768, 12, 12

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__(); self.weight = nn.Parameter(torch.ones(dim)); self.eps = eps
    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight

def rotary_freqs(dim, max_seq, base=10000.0, device="cpu"):
    inv = 1.0 / (base ** (torch.arange(0, dim, 2, device=device).float() / dim))
    f = torch.outer(torch.arange(max_seq, device=device).float(), inv)
    return torch.cos(f), torch.sin(f)

def apply_rotary(x, cos, sin):
    d = x.shape[-1]; x1, x2 = x[..., :d // 2], x[..., d // 2:]
    cos = cos[:x.shape[-2], :d // 2][None, None]; sin = sin[:x.shape[-2], :d // 2][None, None]
    return torch.cat([x1 * cos - x2 * sin, x2 * cos + x1 * sin], -1)

class SwiGLU(nn.Module):
    def __init__(self, dim, hidden):
        super().__init__()
        self.w1 = nn.Linear(dim, hidden, bias=False); self.w2 = nn.Linear(dim, hidden, bias=False)
        self.w3 = nn.Linear(hidden, dim, bias=False)
    def forward(self, x): return self.w3(F.silu(self.w1(x)) * self.w2(x))

class CausalSelfAttention(nn.Module):
    def __init__(self, dim, n_heads, max_seq):
        super().__init__()
        self.n_heads, self.hd = n_heads, dim // n_heads
        self.qkv = nn.Linear(dim, 3 * dim, bias=False); self.proj = nn.Linear(dim, dim, bias=False)
        cos, sin = rotary_freqs(self.hd, max_seq)
        self.register_buffer("cos", cos, persistent=False); self.register_buffer("sin", sin, persistent=False)
    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x).view(B, T, 3, self.n_heads, self.hd).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        q, k = apply_rotary(q, self.cos, self.sin), apply_rotary(k, self.cos, self.sin)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.proj(out.transpose(1, 2).reshape(B, T, C))

class DecoderBlock(nn.Module):
    def __init__(self, dim, n_heads, max_seq, mlp_mult=4):
        super().__init__()
        self.norm1 = RMSNorm(dim); self.attn = CausalSelfAttention(dim, n_heads, max_seq)
        self.norm2 = RMSNorm(dim); self.mlp = SwiGLU(dim, int(dim * mlp_mult * 2 / 3))
    def forward(self, x):
        x = x + self.attn(self.norm1(x)); return x + self.mlp(self.norm2(x))

class DecoderOnlyLM(nn.Module):
    def __init__(self, vocab=VOCAB_SIZE, dim=768, n_layers=12, n_heads=12, max_seq=2048):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab, dim)
        self.blocks = nn.ModuleList([DecoderBlock(dim, n_heads, max_seq) for _ in range(n_layers)])
        self.norm_f = RMSNorm(dim); self.head = nn.Linear(dim, vocab, bias=False)
        self.head.weight = self.tok_emb.weight; self.max_seq = max_seq

class Attn(nn.Module):
    def __init__(self, dim, n_heads, max_seq, n_kv_heads=None, causal=True, window=None, rope=True):
        super().__init__()
        self.h, self.kv = n_heads, (n_kv_heads or n_heads); self.hd = dim // n_heads
        self.causal, self.window, self.rope = causal, window, rope
        self.q = nn.Linear(dim, self.h * self.hd, bias=False); self.k = nn.Linear(dim, self.kv * self.hd, bias=False)
        self.v = nn.Linear(dim, self.kv * self.hd, bias=False); self.proj = nn.Linear(self.h * self.hd, dim, bias=False)
        if rope:
            cos, sin = rotary_freqs(self.hd, max_seq)
            self.register_buffer("cos", cos, persistent=False); self.register_buffer("sin", sin, persistent=False)
    def _mask(self, T, device):
        i = torch.arange(T, device=device)[:, None]; j = torch.arange(T, device=device)[None, :]
        return (j <= i) & (i - j < self.window)
    def forward(self, x, ctx=None):
        B, T, C = x.shape; src = x if ctx is None else ctx; S = src.size(1)
        q = self.q(x).view(B, T, self.h, self.hd).transpose(1, 2)
        k = self.k(src).view(B, S, self.kv, self.hd).transpose(1, 2)
        v = self.v(src).view(B, S, self.kv, self.hd).transpose(1, 2)
        if self.rope and ctx is None:
            q, k = apply_rotary(q, self.cos, self.sin), apply_rotary(k, self.cos, self.sin)
        if self.kv != self.h:
            r = self.h // self.kv; k, v = k.repeat_interleave(r, dim=1), v.repeat_interleave(r, dim=1)
        if ctx is None and self.causal and self.window:
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=self._mask(T, x.device))
        else:
            out = F.scaled_dot_product_attention(q, k, v, is_causal=(self.causal and ctx is None))
        return self.proj(out.transpose(1, 2).reshape(B, T, self.h * self.hd))

class EncBlock(nn.Module):
    def __init__(self, dim, nh, ms, mlp_mult=4):
        super().__init__()
        self.n1, self.attn = RMSNorm(dim), Attn(dim, nh, ms, causal=False)
        self.n2, self.mlp = RMSNorm(dim), SwiGLU(dim, int(dim * mlp_mult * 2 / 3))
    def forward(self, x):
        x = x + self.attn(self.n1(x)); return x + self.mlp(self.n2(x))

class CrossDecBlock(nn.Module):
    def __init__(self, dim, nh, ms, mlp_mult=4):
        super().__init__()
        self.n1, self.self_attn = RMSNorm(dim), Attn(dim, nh, ms, causal=True)
        self.n2, self.cross = RMSNorm(dim), Attn(dim, nh, ms, causal=False, rope=False)
        self.n3, self.mlp = RMSNorm(dim), SwiGLU(dim, int(dim * mlp_mult * 2 / 3))
    def forward(self, x, enc):
        x = x + self.self_attn(self.n1(x)); x = x + self.cross(self.n2(x), ctx=enc); return x + self.mlp(self.n3(x))

class EncoderDecoderLM(nn.Module):
    def __init__(self, vocab=VOCAB_SIZE, dim=768, n_enc=6, n_dec=6, n_heads=12, max_seq=2048):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab, dim)
        self.enc = nn.ModuleList([EncBlock(dim, n_heads, max_seq) for _ in range(n_enc)])
        self.dec = nn.ModuleList([CrossDecBlock(dim, n_heads, max_seq) for _ in range(n_dec)])
        self.norm_e, self.norm_f = RMSNorm(dim), RMSNorm(dim)
        self.head = nn.Linear(dim, vocab, bias=False); self.head.weight = self.tok_emb.weight

class GatedLinearRecurrence(nn.Module):
    def __init__(self, dim, chunk=64):
        super().__init__()
        self.inp = nn.Linear(dim, 4 * dim, bias=False)
        self.conv = nn.Conv1d(dim, dim, 4, groups=dim, padding=3)
        self.out = nn.Linear(dim, dim, bias=False); self.chunk = chunk
    def _scan(self, log_a, z):
        B, T, C = z.shape; L = self.chunk; pad = (-T) % L
        if pad: log_a, z = F.pad(log_a, (0, 0, 0, pad)), F.pad(z, (0, 0, 0, pad))
        n = (T + pad) // L
        log_a, z = log_a.reshape(B, n, L, C), z.reshape(B, n, L, C)
        A = log_a.cumsum(2); local = torch.exp(A) * torch.cumsum(torch.exp(-A) * z, 2)
        prev, outs = z.new_zeros(B, C), []
        for j in range(n):
            o = local[:, j] + torch.exp(A[:, j]) * prev[:, None]; prev = o[:, -1]; outs.append(o)
        return torch.stack(outs, 1).reshape(B, n * L, C)[:, :T]
    def forward(self, x):
        B, T, C = x.shape
        u, ga, gi, go = self.inp(x).chunk(4, -1)
        u = self.conv(u.transpose(1, 2))[..., :T].transpose(1, 2)
        log_a = -0.05 * torch.sigmoid(ga.float()); z = (torch.sigmoid(gi) * F.silu(u)).float()
        h = self._scan(log_a, z).to(x.dtype)
        return self.out(h * F.silu(go))

class Block(nn.Module):
    def __init__(self, dim, mixer, mlp_mult=4):
        super().__init__()
        self.norm1, self.mixer = RMSNorm(dim), mixer
        self.norm2, self.mlp = RMSNorm(dim), SwiGLU(dim, int(dim * mlp_mult * 2 / 3))
    def forward(self, x):
        x = x + self.mixer(self.norm1(x)); return x + self.mlp(self.norm2(x))

class GenericLM(nn.Module):
    def __init__(self, make_block, vocab=VOCAB_SIZE, dim=768, n_layers=12, max_seq=2048):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab, dim)
        self.blocks = nn.ModuleList([make_block(i) for i in range(n_layers)])
        self.norm_f = RMSNorm(dim); self.head = nn.Linear(dim, vocab, bias=False)
        self.head.weight = self.tok_emb.weight

def make_hybrid(dim=768, n_heads=12, max_seq=2048, attn_every=3):
    def f(i):
        mixer = Attn(dim, n_heads, max_seq) if (i + 1) % attn_every == 0 else GatedLinearRecurrence(dim)
        return Block(dim, mixer)
    return f

def make_efficient(dim=768, n_heads=12, n_kv=4, window=256, global_every=4, max_seq=2048):
    def f(i):
        g = (i + 1) % global_every == 0
        return Block(dim, Attn(dim, n_heads, max_seq, n_kv_heads=n_kv, window=None if g else window), mlp_mult=3)
    return f

def build(name):
    d, L, h = DIM, N_LAYERS, N_HEADS
    if name == "A_decoder":   return DecoderOnlyLM(dim=d, n_layers=L, n_heads=h)
    if name == "B_encdec":    return EncoderDecoderLM(dim=d, n_enc=L // 2, n_dec=L // 2, n_heads=h)
    if name == "C_hybrid":    return GenericLM(make_hybrid(dim=d, n_heads=h), dim=d, n_layers=L)
    if name == "D_efficient": return GenericLM(make_efficient(dim=d, n_heads=h, n_kv=max(1, h // 3)), dim=d, n_layers=L)
    raise ValueError(name)

In [6]:
from tokenizers import Tokenizer

class DecBB(nn.Module):          # A / C / D
    def __init__(s, m): super().__init__(); s.tok_emb, s.blocks, s.norm_f = m.tok_emb, m.blocks, m.norm_f
    def forward(s, x, mask=None):
        h = s.tok_emb(x)
        for b in s.blocks: h = b(h)
        return s.norm_f(h)

class EncBB(nn.Module):          # B: encoder only
    def __init__(s, m): super().__init__(); s.tok_emb, s.enc, s.norm_e = m.tok_emb, m.enc, m.norm_e
    def forward(s, x, mask=None):
        h = s.tok_emb(x)
        for b in s.enc: h = b(h)
        return s.norm_e(h)

def _find(pat):
    hits = sorted(f for f in glob.glob(pat, recursive=True) if "_last" not in f)
    assert hits, f"NOT FOUND: {pat}  -> attach your Phase 1 checkpoint dataset to this notebook"
    return hits[0]

def load_tok():
    h = glob.glob(f"{INPUT_ROOT}/**/bpe32k.json", recursive=True) + glob.glob("/kaggle/working/**/bpe32k.json", recursive=True)
    assert h, "bpe32k.json not found -> add it to your Phase 1 Kaggle dataset"
    print("tokenizer:", h[0]); return Tokenizer.from_file(h[0])

def get_phase1(name):
    m = build(name)
    path = _find(f"{INPUT_ROOT}/**/*{name}.pt")
    sd = torch.load(path, map_location="cpu")
    if isinstance(sd, dict) and "model" in sd: sd = sd["model"]
    m.load_state_dict(sd)
    bb = (EncBB if name == "B_encdec" else DecBB)(m).float()      # fp16 ckpt -> fp32 for GradScaler
    return bb, DIM

for _n in ["A_decoder", "B_encdec", "C_hybrid", "D_efficient"]:
    _bb, _H = get_phase1(_n)
    with torch.no_grad(): _o = _bb(torch.randint(0, VOCAB_SIZE, (1, 16)), None)
    print(_n, "OK", tuple(_o.shape), f"{sum(p.numel() for p in _bb.parameters())/1e6:.1f}M backbone params")
    del _bb

A_decoder OK (1, 16, 768) 109.5M backbone params
B_encdec OK (1, 16, 768) 67.1M backbone params
C_hybrid OK (1, 16, 768) 114.3M backbone params
D_efficient OK (1, 16, 768) 85.9M backbone params


In [7]:
TOK = load_tok()
def encode_all(data):
    out, texts = [], [x["text"] for x in data]
    for i in range(0, len(texts), 32):
        out += [np.array(e.ids, dtype=np.uint16) for e in TOK.encode_batch(texts[i:i + 32])]
    return out
t0 = time.time()
ENC = dict(train=encode_all(train), dev=encode_all(dev), test=encode_all(test))
print(f"tokenised in {time.time()-t0:.0f}s")
_l = np.array([len(a) for a in ENC["train"]])
print("train tokens/case: median", int(np.median(_l)), "p90", int(np.percentile(_l, 90)),
      "| cases longer than the chunk budget:", f"{(_l > CHUNK_LEN * MAX_CHUNKS).mean():.0%}")

def chunk_ids(ids, mode="full"):
    L, S, M, n = CHUNK_LEN, STRIDE, MAX_CHUNKS, len(ids)
    if n <= L: return [ids]
    if mode == "head":
        st = list(range(0, n - L + 1, L))[:M]
    elif mode == "tail":
        st = sorted(n - L * (k + 1) for k in range(M) if n - L * (k + 1) >= 0)
    else:                                    # overlapping windows; 1/3 head + 2/3 tail budget
        st = list(range(0, n - L + 1, S))
        if st[-1] + L < n: st.append(n - L)
        if len(st) > M:
            a = M // 3; st = st[:a] + st[-(M - a):]
    return [ids[s:s + L] for s in st]

class DocDS(torch.utils.data.Dataset):
    def __init__(self, data, enc, mode="full"): self.d, self.e, self.mode = data, enc, mode
    def __len__(self): return len(self.d)
    def __getitem__(self, i):
        ch = [c if len(c) else np.array([UNK_ID], dtype=np.uint16) for c in chunk_ids(self.e[i], self.mode)]
        lab = self.d[i]["label"]
        return self.d[i]["id"], ch, (-1 if lab is None else int(lab))

def collate(batch):
    ids = [b[0] for b in batch]; y = torch.tensor([b[2] for b in batch])
    flat = [c for b in batch for c in b[1]]; n = [len(b[1]) for b in batch]
    T = max(len(c) for c in flat)
    X = torch.full((len(flat), T), PAD_ID, dtype=torch.long); M = torch.zeros(len(flat), T, dtype=torch.long)
    for i, c in enumerate(flat):
        X[i, :len(c)] = torch.from_numpy(c.astype(np.int64)); M[i, :len(c)] = 1
    return ids, X, M, n, y

DATA = dict(train=train, dev=dev, test=test)
def make_loader(split, mode="full", shuffle=False, bs=None):
    bs = bs or (BATCH_DOCS if split == "train" else 4)
    g = torch.Generator().manual_seed(SEED) if shuffle else None
    return torch.utils.data.DataLoader(DocDS(DATA[split], ENC[split], mode), batch_size=bs, shuffle=shuffle,
                                       collate_fn=collate, num_workers=NUM_WORKERS, generator=g)
def make_loaders(mode="full"):
    return make_loader("train", mode, True), make_loader("dev", mode), make_loader("test", mode)
print("pipeline ready")

tokenizer: /kaggle/input/datasets/samswe/bpefile/bpe32k.json
tokenised in 70s
train tokens/case: median 2546 p90 7679 | cases longer than the chunk budget: 70%
pipeline ready


In [8]:
class DocClassifier(nn.Module):
    def __init__(self, backbone, H, pooling="mean", n_cls=2):
        super().__init__()
        self.bb, self.pooling = backbone, pooling
        self.att = nn.Linear(H, 1)
        self.head = nn.Sequential(nn.Dropout(0.1), nn.Linear(H, n_cls))
    def chunk_rep(self, X, M):
        h = self.bb(X, M).float()
        m = M.unsqueeze(-1).to(h.dtype)
        return (h * m).sum(1) / m.sum(1).clamp(min=1)
    def forward(self, X, M, n):
        r = self.chunk_rep(X, M); outs = []; s = 0
        for k in n:
            c = r[s:s + k]; s += k
            if self.pooling == "attn":
                w = torch.softmax(self.att(c).squeeze(-1), 0); outs.append((w.unsqueeze(-1) * c).sum(0))
            else:
                outs.append(c.mean(0))
        return self.head(torch.stack(outs))

class LoRMALinear(nn.Module):
    """Low-rank MULTIPLICATIVE adaptation: y = (I + B A) (W x). B zero-init -> starts at the pretrained model."""
    def __init__(self, base, r=8):
        super().__init__()
        self.base = base; o = base.out_features
        self.A = nn.Parameter(torch.randn(r, o) * 0.01); self.B = nn.Parameter(torch.zeros(o, r))
    def forward(self, x):
        h = F.linear(x, self.base.weight)
        return h + (h @ self.A.t()) @ self.B.t()

class Adapter(nn.Module):
    def __init__(self, base, r=32):
        super().__init__()
        self.base = base; o = base.out_features
        self.down, self.up = nn.Linear(o, r), nn.Linear(r, o)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)
    def forward(self, x):
        h = self.base(x); return h + self.up(F.gelu(self.down(h)))

ATTN_LINEARS = {"qkv", "q", "k", "v", "proj", "inp", "out"}
def wrap_linears(model, cls, r):
    for mod in list(model.modules()):
        for cn, ch in list(mod.named_children()):
            if isinstance(ch, nn.Linear) and cn in ATTN_LINEARS:
                setattr(mod, cn, cls(ch, r))

def apply_technique(clf, tech, r=8, last_k=4):
    for p in clf.parameters(): p.requires_grad = False
    if tech == "full":
        for p in clf.parameters(): p.requires_grad = True
    elif tech == "partial":
        stack = list(clf.bb.blocks) if hasattr(clf.bb, "blocks") else list(clf.bb.enc)
        for b in stack[-last_k:]:
            for p in b.parameters(): p.requires_grad = True
    elif tech == "bitfit":                                       # biases + RMSNorm scales
        for n, p in clf.bb.named_parameters():
            if n.endswith("bias"): p.requires_grad = True
        for m in clf.bb.modules():
            if isinstance(m, RMSNorm): m.weight.requires_grad = True
    elif tech == "lorma":
        wrap_linears(clf.bb, LoRMALinear, r)
    elif tech == "adapter":
        wrap_linears(clf.bb, Adapter, 32)
    for n, p in clf.bb.named_parameters():
        if tech == "lorma" and n.split(".")[-1] in ("A", "B"): p.requires_grad = True
        if tech == "adapter" and (".down." in n or ".up." in n): p.requires_grad = True
    for p in list(clf.head.parameters()) + list(clf.att.parameters()): p.requires_grad = True
    tr = sum(p.numel() for p in clf.parameters() if p.requires_grad); tot = sum(p.numel() for p in clf.parameters())
    print(f"  {tech}: trainable {tr/1e6:.2f}M / {tot/1e6:.2f}M ({100*tr/tot:.2f}%)")
    return tr, tot

In [9]:
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix

STATE = f"{OUT}/state.json"
RESULTS, PROBS = {}, {}
if os.path.exists(STATE):
    _d = json.load(open(STATE)); RESULTS, PROBS = _d["results"], _d["probs"]
    print("resumed finished runs:", list(RESULTS))
def save_state(): json.dump(dict(results=RESULTS, probs=PROBS), open(STATE, "w"))

RUNS = {
    "A_lorma":  dict(phase1="A_decoder",   tech="lorma",  pooling="mean", lr=LR["lorma"]),
    "B_full":   dict(phase1="B_encdec",    tech="full",   pooling="attn", lr=LR["full"]),
    "C_bitfit": dict(phase1="C_hybrid",    tech="bitfit", pooling="mean", lr=LR["bitfit"]),
    "D_full":   dict(phase1="D_efficient", tech="full",   pooling="mean", lr=LR["full"]),
    "D_kd":     dict(phase1="D_efficient", tech="full",   pooling="mean", lr=LR["kd"]),
}
YDEV = np.array([x["label"] for x in dev])

def metrics(y, pred):
    return dict(macroF1=f1_score(y, pred, average="macro"), P=precision_score(y, pred, average="macro", zero_division=0),
                R=recall_score(y, pred, average="macro", zero_division=0), acc=float((y == pred).mean()))
def best_thresh(p, y):
    ts = np.linspace(0.05, 0.95, 91); f = [f1_score(y, (p > t).astype(int), average="macro") for t in ts]
    return float(ts[int(np.argmax(f))]), float(max(f))
def class_weights():
    c = np.bincount([x["label"] for x in train], minlength=2); return torch.tensor(c.sum() / (2.0 * c), dtype=torch.float)

@torch.no_grad()
def predict(clf, loader):
    clf.eval(); P, Y, I = [], [], []
    for ids, X, M, n, y in loader:
        with AC(): lg = clf(X.to(DEV), M.to(DEV), n)
        P.append(torch.softmax(lg.float(), -1)[:, 1].cpu()); Y.append(y); I += ids
    return I, torch.cat(P).numpy(), torch.cat(Y).numpy()

def fit(clf, trl, dvl, lr, epochs, teacher=None, T=2.0, alpha=0.5, tag="", weighted=True):
    cw = class_weights().to(DEV) if weighted else None
    hd = [p for n, p in clf.named_parameters() if p.requires_grad and n.startswith(("head.", "att."))]
    bbp = [p for n, p in clf.named_parameters() if p.requires_grad and not n.startswith(("head.", "att."))]
    groups = [dict(params=hd, lr=HEAD_LR)] + ([dict(params=bbp, lr=lr)] if bbp else [])
    opt = torch.optim.AdamW(groups, weight_decay=0.01)
    spe = math.ceil(len(trl) / GRAD_ACCUM); total = epochs * spe
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[g["lr"] for g in groups], total_steps=total, pct_start=0.1)
    try: scaler = torch.amp.GradScaler("cuda", enabled=(DEV == "cuda"))
    except Exception: scaler = torch.cuda.amp.GradScaler(enabled=(DEV == "cuda"))
    params = hd + bbp; best = (-1.0, None, 0.5); t_start = time.time()
    for ep in range(epochs):
        clf.train(); opt.zero_grad(set_to_none=True); tl = 0.0; t0 = time.time()
        for i, (ids, X, M, n, y) in enumerate(trl):
            X, M, y = X.to(DEV), M.to(DEV), y.to(DEV)
            with AC():
                lg = clf(X, M, n).float()
                loss = F.cross_entropy(lg, y, weight=cw)
                if teacher is not None:
                    with torch.no_grad(): tlg = teacher(X, M, n).float()
                    kd = F.kl_div(F.log_softmax(lg / T, -1), F.softmax(tlg / T, -1), reduction="batchmean") * T * T
                    loss = alpha * loss + (1 - alpha) * kd
            scaler.scale(loss / GRAD_ACCUM).backward(); tl += loss.item()
            if (i + 1) % GRAD_ACCUM == 0 or i + 1 == len(trl):
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, 1.0)
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sch.step()
            if i % (5 if SMOKE else 50) == 0:
                el = time.time() - t0; print(f"  [{tag}] ep{ep+1} batch {i}/{len(trl)} loss {loss.item():.3f} | ep ETA {el/(i+1)*(len(trl)-i-1)/60:.1f} min", flush=True)
        I, p, y = predict(clf, dvl); th, f = best_thresh(p, y)
        print(f"[{tag}] ep{ep+1} loss {tl/len(trl):.4f} | dev macroF1@0.5 {metrics(y, (p > .5).astype(int))['macroF1']:.4f} | best {f:.4f}@{th:.2f} | {(time.time()-t0)/60:.1f} min", flush=True)
        if f > best[0]:
            best = (f, {n: p_.detach().cpu().clone() for n, p_ in clf.named_parameters() if p_.requires_grad}, th)
            torch.save(dict(w=best[1], th=th, f1=f), f"{OUT}/{tag}_best_partial.pt")
    named = dict(clf.named_parameters())
    with torch.no_grad():
        for n, v in best[1].items(): named[n].copy_(v.to(named[n].device))
    print(f"[{tag}] best dev macroF1 {best[0]:.4f} @ th {best[2]:.2f} | total {(time.time()-t_start)/60:.1f} min")
    return best[0], best[2]

@torch.no_grad()
def efficiency(clf, loader):
    n = sum(p.numel() for p in clf.parameters()); trn = sum(p.numel() for p in clf.parameters() if p.requires_grad)
    if DEV == "cuda": torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize()
    clf.eval(); t0 = time.time(); nd = 0
    for i, (ids, X, M, n_, y) in enumerate(loader):
        with AC(): clf(X.to(DEV), M.to(DEV), n_)
        nd += len(ids)
        if i == 4: break
    if DEV == "cuda": torch.cuda.synchronize()
    return dict(params_M=n / 1e6, trainable_M=trn / 1e6,
                peak_mem_GB=(torch.cuda.max_memory_allocated() / 1e9 if DEV == "cuda" else 0.0), sec_per_case=(time.time() - t0) / max(nd, 1))

def write_zip(tag, ids, preds):
    assert len(ids) == len(test) and set(ids) == set(x["id"] for x in test), "ids do not match the official test ids"
    d = f"{OUT}/sub_{tag}"; os.makedirs(d, exist_ok=True)
    json.dump([{ID_KEY: i, PRED_KEY: int(p)} for i, p in zip(ids, preds)], open(f"{d}/prediction.json", "w"))
    with zipfile.ZipFile(f"{OUT}/{tag}.zip", "w", zipfile.ZIP_DEFLATED) as z: z.write(f"{d}/prediction.json", arcname="prediction.json")
    print(f"  wrote {OUT}/{tag}.zip | positives {int(np.sum(preds))}/{len(preds)}")

def run_experiment(tag, cfg, teacher=None, weighted=True, submit=True, epochs=None, mode="full", save_model=True):
    T0 = time.time()
    def say(msg): print(f"  >> [{tag}] {msg} (+{time.time()-T0:.0f}s)", flush=True)
    if tag in RESULTS:
        print(f"skip {tag} (already done): macroF1 {RESULTS[tag]['macroF1']:.4f}")
        if submit and not os.path.exists(f"{OUT}/{tag}.zip"):
            write_zip(tag, PROBS[tag]["test_ids"], (np.array(PROBS[tag]["test"]) > RESULTS[tag]["th"]).astype(int))
        return RESULTS[tag]
    epochs = epochs or EPOCHS; seed_all(); print("=" * 18, tag, cfg)
    say("STARTED: loading Phase 1 checkpoint")
    bb, H = get_phase1(cfg["phase1"]); clf = DocClassifier(bb, H, cfg["pooling"])
    say("checkpoint loaded, applying technique")
    apply_technique(clf, cfg["tech"]); clf.to(DEV)
    say(f"model on {DEV}, building loaders")
    trl, dvl, tel = make_loaders(mode)
    say(f"training starts: {len(trl)} batches/epoch, {epochs} epoch(s)")
    f, th = fit(clf, trl, dvl, cfg["lr"], epochs, teacher=teacher, tag=tag, weighted=weighted)
    say("training done, predicting dev/test")
    Id, pdv, yd = predict(clf, dvl); It, pts, _ = predict(clf, tel)
    say("predictions done, computing efficiency stats")
    res = dict(metrics(yd, (pdv > th).astype(int)), th=th, tech=cfg["tech"], phase1=cfg["phase1"], pooling=cfg["pooling"],
               weighted=weighted, epochs=epochs, **efficiency(clf, dvl))
    RESULTS[tag] = res; PROBS[tag] = dict(dev=pdv.tolist(), test=pts.tolist(), dev_ids=Id, test_ids=It)
    if submit: write_zip(tag, It, (pts > th).astype(int))
    if save_model:
        torch.save({k: (v.half() if v.is_floating_point() else v) for k, v in clf.state_dict().items()}, f"{OUT}/{tag}.pt")
    save_state(); say("FINISHED")
    print(tag, {k: (round(v, 4) if isinstance(v, float) else v) for k, v in res.items()})
    del clf, bb, trl, dvl, tel; gc.collect()
    if DEV == "cuda": torch.cuda.empty_cache()
    return res

def load_clf(tag):
    c = RUNS[tag]; bb, H = get_phase1(c["phase1"]); clf = DocClassifier(bb, H, c["pooling"])
    apply_technique(clf, c["tech"]); clf.load_state_dict(torch.load(f"{OUT}/{tag}.pt", map_location="cpu"))
    return clf.float().to(DEV).eval()
print("utilities ready")

utilities ready


In [10]:
run_experiment("A_lorma", RUNS["A_lorma"])
# OOM? set RUNS["B_full"]["tech"] = "partial" (last 4 encoder blocks only)
run_experiment("B_full", RUNS["B_full"])
run_experiment("C_bitfit", RUNS["C_bitfit"])


================== A_lorma {'phase1': 'A_decoder', 'tech': 'lorma', 'pooling': 'mean', 'lr': 0.0005}
  >> [A_lorma] STARTED: loading Phase 1 checkpoint (+0s)
  >> [A_lorma] checkpoint loaded, applying technique (+1s)
  lorma: trainable 0.59M / 110.12M (0.54%)
  >> [A_lorma] model on cuda, building loaders (+1s)
  >> [A_lorma] training starts: 1500 batches/epoch, 1 epoch(s) (+1s)
  [A_lorma] ep1 batch 0/1500 loss 0.636 | ep ETA 28.0 min
  [A_lorma] ep1 batch 50/1500 loss 0.653 | ep ETA 3.9 min
  [A_lorma] ep1 batch 100/1500 loss 0.940 | ep ETA 3.5 min
  [A_lorma] ep1 batch 150/1500 loss 0.936 | ep ETA 3.3 min
  [A_lorma] ep1 batch 200/1500 loss 0.684 | ep ETA 3.1 min
  [A_lorma] ep1 batch 250/1500 loss 0.761 | ep ETA 3.0 min
  [A_lorma] ep1 batch 300/1500 loss 1.201 | ep ETA 2.9 min
  [A_lorma] ep1 batch 350/1500 loss 1.163 | ep ETA 2.7 min
  [A_lorma] ep1 batch 400/1500 loss 0.449 | ep ETA 2.6 min
  [A_lorma] ep1 batch 450/1500 loss 0.499 | ep ETA 2.5 min
  [A_lorma] ep1 batch 500/1500

/tmp/ipykernel_23/1777849108.py:61: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sch.step()


  [B_full] ep1 batch 50/1500 loss 0.606 | ep ETA 2.5 min
  [B_full] ep1 batch 100/1500 loss 0.794 | ep ETA 2.4 min
  [B_full] ep1 batch 150/1500 loss 0.870 | ep ETA 2.3 min
  [B_full] ep1 batch 200/1500 loss 0.344 | ep ETA 2.3 min
  [B_full] ep1 batch 250/1500 loss 0.719 | ep ETA 2.2 min
  [B_full] ep1 batch 300/1500 loss 1.145 | ep ETA 2.1 min
  [B_full] ep1 batch 350/1500 loss 0.836 | ep ETA 2.0 min
  [B_full] ep1 batch 400/1500 loss 0.753 | ep ETA 1.9 min
  [B_full] ep1 batch 450/1500 loss 0.767 | ep ETA 1.8 min
  [B_full] ep1 batch 500/1500 loss 1.066 | ep ETA 1.7 min
  [B_full] ep1 batch 550/1500 loss 0.787 | ep ETA 1.7 min
  [B_full] ep1 batch 600/1500 loss 0.428 | ep ETA 1.6 min
  [B_full] ep1 batch 650/1500 loss 0.596 | ep ETA 1.5 min
  [B_full] ep1 batch 700/1500 loss 0.517 | ep ETA 1.4 min
  [B_full] ep1 batch 750/1500 loss 0.404 | ep ETA 1.3 min
  [B_full] ep1 batch 800/1500 loss 0.882 | ep ETA 1.2 min
  [B_full] ep1 batch 850/1500 loss 0.820 | ep ETA 1.1 min
  [B_full] ep1 

{'macroF1': 0.5702655225656212,
 'P': 0.5705256073432584,
 'R': 0.5704225352112675,
 'acc': 0.5704225352112676,
 'th': 0.41999999999999993,
 'tech': 'bitfit',
 'phase1': 'C_hybrid',
 'pooling': 'mean',
 'weighted': True,
 'epochs': 1,
 'params_M': 114.281475,
 'trainable_M': 0.027651,
 'peak_mem_GB': 0.680246272,
 'sec_per_case': 0.03812919855117798}

In [11]:
if "D_kd" in RESULTS:
    run_experiment("D_kd", RUNS["D_kd"])
else:
    sp = sum(p.numel() for p in DocClassifier(DecBB(build("D_efficient")), DIM).parameters()) / 1e6
    cand = [t for t in ("A_lorma", "B_full", "C_bitfit") if t in RESULTS]
    assert cand, "run Cells 9-11 first"
    bigger = [t for t in cand if RESULTS[t]["params_M"] > sp]
    pool = bigger or cand
    if not bigger: print("WARNING: no teacher is larger than the student; using best overall (mention in report)")
    tname = max(pool, key=lambda t: RESULTS[t]["macroF1"])
    print(f"student D_efficient {sp:.1f}M | teacher {tname} {RESULTS[tname]['params_M']:.1f}M, dev macroF1 {RESULTS[tname]['macroF1']:.4f}")
    teacher = load_clf(tname)
    for p_ in teacher.parameters(): p_.requires_grad = False
    run_experiment("D_kd", RUNS["D_kd"], teacher=teacher)
    RESULTS["D_kd"]["teacher"] = tname; save_state()
    del teacher; gc.collect()
    if DEV == "cuda": torch.cuda.empty_cache()

student D_efficient 85.9M | teacher C_bitfit 114.3M, dev macroF1 0.5703
  bitfit: trainable 0.03M / 114.28M (0.02%)
================== D_kd {'phase1': 'D_efficient', 'tech': 'full', 'pooling': 'mean', 'lr': 0.0001}
  >> [D_kd] STARTED: loading Phase 1 checkpoint (+0s)
  >> [D_kd] checkpoint loaded, applying technique (+1s)
  full: trainable 85.94M / 85.94M (100.00%)
  >> [D_kd] model on cuda, building loaders (+1s)
  >> [D_kd] training starts: 1500 batches/epoch, 1 epoch(s) (+1s)
  [D_kd] ep1 batch 0/1500 loss 0.332 | ep ETA 10.5 min
  [D_kd] ep1 batch 50/1500 loss 0.463 | ep ETA 5.7 min
  [D_kd] ep1 batch 100/1500 loss 0.338 | ep ETA 5.5 min
  [D_kd] ep1 batch 150/1500 loss 0.417 | ep ETA 5.3 min
  [D_kd] ep1 batch 200/1500 loss 0.308 | ep ETA 5.1 min
  [D_kd] ep1 batch 250/1500 loss 0.390 | ep ETA 4.9 min
  [D_kd] ep1 batch 300/1500 loss 0.597 | ep ETA 4.7 min
  [D_kd] ep1 batch 350/1500 loss 0.414 | ep ETA 4.5 min
  [D_kd] ep1 batch 400/1500 loss 0.322 | ep ETA 4.3 min
  [D_kd] ep1 

In [12]:
run_experiment("D_full", RUNS["D_full"])

================== D_full {'phase1': 'D_efficient', 'tech': 'full', 'pooling': 'mean', 'lr': 0.0001}
  >> [D_full] STARTED: loading Phase 1 checkpoint (+0s)
  >> [D_full] checkpoint loaded, applying technique (+1s)
  full: trainable 85.94M / 85.94M (100.00%)
  >> [D_full] model on cuda, building loaders (+1s)
  >> [D_full] training starts: 1500 batches/epoch, 1 epoch(s) (+1s)
  [D_full] ep1 batch 0/1500 loss 0.627 | ep ETA 6.5 min
  [D_full] ep1 batch 50/1500 loss 0.906 | ep ETA 4.1 min
  [D_full] ep1 batch 100/1500 loss 0.427 | ep ETA 4.0 min
  [D_full] ep1 batch 150/1500 loss 1.855 | ep ETA 3.8 min
  [D_full] ep1 batch 200/1500 loss 0.940 | ep ETA 3.7 min
  [D_full] ep1 batch 250/1500 loss 0.550 | ep ETA 3.5 min
  [D_full] ep1 batch 300/1500 loss 0.820 | ep ETA 3.4 min
  [D_full] ep1 batch 350/1500 loss 0.986 | ep ETA 3.3 min
  [D_full] ep1 batch 400/1500 loss 0.771 | ep ETA 3.1 min
  [D_full] ep1 batch 450/1500 loss 0.541 | ep ETA 3.0 min
  [D_full] ep1 batch 500/1500 loss 1.066 | e

{'macroF1': 0.5733560090702948,
 'P': 0.5734989648033126,
 'R': 0.5734406438631791,
 'acc': 0.5734406438631791,
 'th': 0.42999999999999994,
 'tech': 'full',
 'phase1': 'D_efficient',
 'pooling': 'mean',
 'weighted': True,
 'epochs': 1,
 'params_M': 85.939203,
 'trainable_M': 85.939203,
 'peak_mem_GB': 0.620942848,
 'sec_per_case': 0.03074016571044922}

In [13]:
tags = [t for t in ("A_lorma", "B_full", "C_bitfit", "D_kd", "D_full") if t in PROBS]
assert len(tags) >= 2, "need at least 2 finished runs"
for t in tags: assert PROBS[t]["dev_ids"] == PROBS[tags[0]]["dev_ids"] and PROBS[t]["test_ids"] == PROBS[tags[0]]["test_ids"]
def ens_eval(ts):
    p = np.mean([PROBS[t]["dev"] for t in ts], 0); th, f = best_thresh(p, YDEV); return f, th
ranked = sorted(tags, key=lambda t: -RESULTS[t]["macroF1"])
cands = {"all": tags, "top3": ranked[:3], "top2": ranked[:2]}
sc = {k: ens_eval(v) for k, v in cands.items()}; print({k: (round(f, 4), round(t, 2)) for k, (f, t) in sc.items()})
bk = max(sc, key=lambda k: sc[k][0]); f, th = sc[bk]; ts = cands[bk]
pdv = np.mean([PROBS[t]["dev"] for t in ts], 0); pts = np.mean([PROBS[t]["test"] for t in ts], 0)
RESULTS["E_ensemble"] = dict(metrics(YDEV, (pdv > th).astype(int)), th=th, members=ts)
PROBS["E_ensemble"] = dict(dev=pdv.tolist(), test=pts.tolist(), dev_ids=PROBS[tags[0]]["dev_ids"], test_ids=PROBS[tags[0]]["test_ids"])
write_zip("E_ensemble", PROBS[tags[0]]["test_ids"], (pts > th).astype(int)); save_state()
print("ensemble members:", ts, "| dev macroF1", round(f, 4))

{'all': (0.5825, 0.43), 'top3': (0.5794, 0.43), 'top2': (0.5794, 0.43)}
  wrote /kaggle/working/run/E_ensemble.zip | positives 791/1517
ensemble members: ['A_lorma', 'B_full', 'C_bitfit', 'D_kd', 'D_full'] | dev macroF1 0.5825


In [14]:
df = pd.DataFrame(RESULTS).T
for c in df.columns:
    try: df[c] = pd.to_numeric(df[c]).round(4)
    except (ValueError, TypeError): pass
cols = [c for c in ["phase1", "tech", "pooling", "macroF1", "P", "R", "acc", "th", "params_M", "trainable_M", "peak_mem_GB", "sec_per_case", "teacher", "members"] if c in df.columns]
df = df[cols].sort_values("macroF1", ascending=False); print(df.to_string()); df.to_csv(f"{OUT}/results.csv")

                 phase1    tech pooling  macroF1       P       R     acc    th  params_M  trainable_M  peak_mem_GB  sec_per_case   teacher                                    members
B_full         B_encdec    full    attn   0.5912  0.5918  0.5915  0.5915  0.45   67.0556      67.0556       0.5119        0.0190       NaN                                        NaN
D_kd        D_efficient    full    mean   0.5845  0.5845  0.5845  0.5845  0.42   85.9392      85.9392       1.0844        0.0305  C_bitfit                                        NaN
E_ensemble          NaN     NaN     NaN   0.5825  0.5825  0.5825  0.5825  0.43       NaN          NaN          NaN           NaN       NaN  [A_lorma, B_full, C_bitfit, D_kd, D_full]
D_full      D_efficient    full    mean   0.5734  0.5735  0.5734  0.5734  0.43   85.9392      85.9392       0.6209        0.0307       NaN                                        NaN
C_bitfit       C_hybrid  bitfit    mean   0.5703  0.5705  0.5704  0.5704  0.42  114.2815  

In [15]:
singles = [t for t in RESULTS if t in RUNS]
best = max(singles, key=lambda t: RESULTS[t]["macroF1"]); print("best single model:", best)
clf = load_clf(best); th = RESULTS[best]["th"]; rows = {}
for mode in ["head", "tail", "full"]:
    I, p, y = predict(clf, make_loader("dev", mode)); rows[mode] = metrics(y, (p > th).astype(int))
abl = pd.DataFrame(rows).T.round(4); print(abl); abl.to_csv(f"{OUT}/ablation_section.csv")
del clf; gc.collect()
if DEV == "cuda": torch.cuda.empty_cache()

best single model: B_full
  full: trainable 67.06M / 67.06M (100.00%)
      macroF1       P       R     acc
head   0.5734  0.5734  0.5734  0.5734
tail   0.5956  0.6016  0.5986  0.5986
full   0.5913  0.5918  0.5915  0.5915


In [16]:
bt = max([t for t in RESULTS if t in PROBS], key=lambda t: RESULTS[t]["macroF1"]); print("analysing:", bt)
p = np.array(PROBS[bt]["dev"]); th = RESULTS[bt]["th"]; pred = (p > th).astype(int); ids = PROBS[bt]["dev_ids"]
print(confusion_matrix(YDEV, pred)); dm = {x["id"]: x for x in dev}
fp = [i for i, a, b in zip(ids, pred, YDEV) if a == 1 and b == 0]; fn = [i for i, a, b in zip(ids, pred, YDEV) if a == 0 and b == 1]
wl = lambda L: float(np.mean([len(dm[i]["text"].split()) for i in L])) if L else 0.0
print(f"FP {len(fp)} avg words {wl(fp):.0f} | FN {len(fn)} avg words {wl(fn):.0f} | all dev avg words {wl(list(dm)):.0f}")
conf = np.abs(p - th)
print("mean |p-th| correct:", conf[pred == YDEV].mean().round(3), "wrong:", conf[pred != YDEV].mean().round(3))
for i in fp[:2] + fn[:2]: print("\n--", "FP" if i in fp else "FN", i, "...", dm[i]["text"][-500:].replace("\n", " "))
json.dump(dict(fp=[str(i) for i in fp], fn=[str(i) for i in fn]), open(f"{OUT}/error_ids.json", "w"))

analysing: B_full
[[616 378]
 [434 560]]
FP 378 avg words 3440 | FN 434 avg words 3760 | all dev avg words 3752
mean |p-th| correct: 0.007 wrong: 0.006

-- FP 1989_75 ... s a party to the fraudulent act for self aggrandisemen t. he prepared bogus documents for withdrawal of salary in t he name of ashok kumar who was number working in his division. he has thus proved himself unbecoming and unworthy to hold a ny post. any sympathy or charitable view on such officials wi ll number be companyducive to keep the streams of administration pu re which is so vital for the success of our democrary. in the result we allow the appeal and set aside t he order of the tribunal.

-- FP 1959_102 ... at the companypany did  c. as. number. 746 747 of 1957 decided on 29-4-1059. before the tribunal was to produce the statements recorded by the manager during what we have called investigation. this left the matters where they were and das had never an opportunity of questioning the witnesses after knumberin

In [17]:
test_ids = set(x["id"] for x in test); ok_all = True
for z in sorted(glob.glob(f"{OUT}/*.zip")):
    with zipfile.ZipFile(z) as zf:
        names = zf.namelist(); p = json.loads(zf.read("prediction.json"))
    ids = [x[ID_KEY] for x in p]; vals = {x[PRED_KEY] for x in p}
    ok = names == ["prediction.json"] and set(ids) == test_ids and len(ids) == len(test_ids) and vals <= {0, 1}
    ok_all &= ok
    print(("OK  " if ok else "BAD ") + os.path.basename(z), "| files:", names, "| n:", len(ids), "| preds:", sorted(vals), "| pos rate:", round(float(np.mean([x[PRED_KEY] for x in p])), 3))
print("ALL ZIPS VALID" if ok_all else "FIX THE BAD ZIPS ABOVE"); print("SMOKE MODE: do NOT upload these; set SMOKE=False and rerun" if SMOKE else "-> upload the .zip files from " + OUT)

OK  A_lorma.zip | files: ['prediction.json'] | n: 1517 | preds: [0, 1] | pos rate: 0.503
OK  B_full.zip | files: ['prediction.json'] | n: 1517 | preds: [0, 1] | pos rate: 0.471
OK  C_bitfit.zip | files: ['prediction.json'] | n: 1517 | preds: [0, 1] | pos rate: 0.49
OK  D_full.zip | files: ['prediction.json'] | n: 1517 | preds: [0, 1] | pos rate: 0.498
OK  D_kd.zip | files: ['prediction.json'] | n: 1517 | preds: [0, 1] | pos rate: 0.491
OK  E_ensemble.zip | files: ['prediction.json'] | n: 1517 | preds: [0, 1] | pos rate: 0.521
ALL ZIPS VALID
-> upload the .zip files from /kaggle/working/run


In [18]:
base = RUNS["C_bitfit"]
run_experiment("abl_unweighted", dict(base), weighted=False, submit=False, save_model=False)
run_experiment("abl_attn_pool", dict(base, pooling="attn"), submit=False, save_model=False)
abl2 = pd.DataFrame({k: RESULTS[k] for k in ["C_bitfit", "abl_unweighted", "abl_attn_pool"] if k in RESULTS}).T[["pooling", "weighted", "macroF1", "P", "R"]]
print(abl2.to_string())

================== abl_unweighted {'phase1': 'C_hybrid', 'tech': 'bitfit', 'pooling': 'mean', 'lr': 0.002}
  >> [abl_unweighted] STARTED: loading Phase 1 checkpoint (+0s)
  >> [abl_unweighted] checkpoint loaded, applying technique (+1s)
  bitfit: trainable 0.03M / 114.28M (0.02%)
  >> [abl_unweighted] model on cuda, building loaders (+1s)
  >> [abl_unweighted] training starts: 1500 batches/epoch, 1 epoch(s) (+1s)
  [abl_unweighted] ep1 batch 0/1500 loss 0.680 | ep ETA 7.6 min
  [abl_unweighted] ep1 batch 50/1500 loss 0.877 | ep ETA 4.2 min
  [abl_unweighted] ep1 batch 100/1500 loss 0.913 | ep ETA 4.1 min
  [abl_unweighted] ep1 batch 150/1500 loss 0.619 | ep ETA 3.9 min
  [abl_unweighted] ep1 batch 200/1500 loss 0.408 | ep ETA 3.8 min
  [abl_unweighted] ep1 batch 250/1500 loss 0.843 | ep ETA 3.6 min
  [abl_unweighted] ep1 batch 300/1500 loss 1.348 | ep ETA 3.5 min
  [abl_unweighted] ep1 batch 350/1500 loss 0.928 | ep ETA 3.3 min
  [abl_unweighted] ep1 batch 400/1500 loss 0.581 | ep ETA 

In [19]:
# ===== ONE STRONG ZIP: B_encdec, full fine-tune, ALL train data, 6 chunks, 3 epochs =====
assert not SMOKE, "SMOKE must be False"
assert PATHS["dev"], "needs a separate dev file, otherwise training on all data would leak dev"

MAX_CHUNKS = 6          # too slow -> 4 ; out of memory -> 4
BEST_EPOCHS = 3         # shorter -> 2

full_train = norm(read_any(PATHS["train"]))
for x in full_train: x["label"] = LABMAP[x["label"]]
train = full_train
DATA["train"] = train
t0 = time.time(); ENC["train"] = encode_all(train)
print(f"{len(train)} train cases tokenised in {time.time()-t0:.0f}s | labels {np.bincount([x['label'] for x in train])}")

cfg = dict(RUNS["B_full"], lr=1e-4)
run_experiment("BEST_B", cfg, weighted=False, epochs=BEST_EPOCHS)

# validate the zip
z = f"{OUT}/BEST_B.zip"
with zipfile.ZipFile(z) as zf:
    names = zf.namelist(); p = json.loads(zf.read("prediction.json"))
ids = [x[ID_KEY] for x in p]; vals = {x[PRED_KEY] for x in p}
ok = names == ["prediction.json"] and set(ids) == set(x["id"] for x in test) and len(ids) == len(test) and vals <= {0, 1}
print(("VALID " if ok else "BAD ") + z, "| n:", len(ids), "| preds:", sorted(vals), "| pos rate:", round(float(np.mean([x[PRED_KEY] for x in p])), 3))
print("-> submit this file:", z)

37387 train cases tokenised in 377s | labels [22067 15320]
================== BEST_B {'phase1': 'B_encdec', 'tech': 'full', 'pooling': 'attn', 'lr': 0.0001}
  >> [BEST_B] STARTED: loading Phase 1 checkpoint (+0s)
  >> [BEST_B] checkpoint loaded, applying technique (+1s)
  full: trainable 67.06M / 67.06M (100.00%)
  >> [BEST_B] model on cuda, building loaders (+1s)
  >> [BEST_B] training starts: 18694 batches/epoch, 3 epoch(s) (+1s)
  [BEST_B] ep1 batch 0/18694 loss 1.497 | ep ETA 103.7 min


/tmp/ipykernel_23/1777849108.py:61: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sch.step()


  [BEST_B] ep1 batch 50/18694 loss 0.638 | ep ETA 56.2 min
  [BEST_B] ep1 batch 100/18694 loss 1.029 | ep ETA 60.2 min
  [BEST_B] ep1 batch 150/18694 loss 0.588 | ep ETA 58.2 min
  [BEST_B] ep1 batch 200/18694 loss 0.758 | ep ETA 56.6 min
  [BEST_B] ep1 batch 250/18694 loss 0.874 | ep ETA 55.0 min
  [BEST_B] ep1 batch 300/18694 loss 0.377 | ep ETA 54.0 min
  [BEST_B] ep1 batch 350/18694 loss 0.726 | ep ETA 53.3 min
  [BEST_B] ep1 batch 400/18694 loss 0.682 | ep ETA 52.8 min
  [BEST_B] ep1 batch 450/18694 loss 0.831 | ep ETA 52.8 min
  [BEST_B] ep1 batch 500/18694 loss 0.582 | ep ETA 52.8 min
  [BEST_B] ep1 batch 550/18694 loss 0.577 | ep ETA 52.7 min
  [BEST_B] ep1 batch 600/18694 loss 0.694 | ep ETA 52.6 min
  [BEST_B] ep1 batch 650/18694 loss 0.569 | ep ETA 52.7 min
  [BEST_B] ep1 batch 700/18694 loss 0.704 | ep ETA 52.6 min
  [BEST_B] ep1 batch 750/18694 loss 0.669 | ep ETA 52.4 min
  [BEST_B] ep1 batch 800/18694 loss 0.800 | ep ETA 52.1 min
  [BEST_B] ep1 batch 850/18694 loss 0.619